# 00 Environment Setup

**Purpose:** Verify the Databricks environment, install required libraries, confirm MLflow and Delta access, and validate secrets.

Run each cell sequentially on a cluster with Databricks Runtime for Machine Learning (recommended).

In [ ]:
# Install libraries used across the repo (idempotent)
# Run this cell once per cluster session.
%pip install -q "mlflow>=2.0" delta-spark pandas scikit-learn optuna requests scipy

import importlib
packages = ["mlflow","pandas","sklearn","delta","optuna","requests","scipy"]
for p in packages:
    try:
        m = importlib.import_module(p)
        print(p, getattr(m, "__version__", "version unknown"))
    except Exception as e:
        print(p, "not available:", e)

In [ ]:
# Basic Spark / runtime checks
from pyspark.sql import SparkSession
import os
spark = SparkSession.builder.getOrCreate()

print("Spark version:", spark.version)

# Databricks runtime environment variable (may vary by runtime)
db_runtime = os.environ.get("DATABRICKS_RUNTIME_VERSION") or os.environ.get("DATABRICKS_RUNTIME")
print("Databricks runtime env:", db_runtime or "Not detected (non-Databricks runtime)")

import mlflow
print("MLflow tracking URI:", mlflow.get_tracking_uri())

In [ ]:
# Quick Delta write/read smoke test (uses a temporary table)
from pyspark.sql import Row
tmp = spark.createDataFrame([Row(x=1, msg="hello")])
tmp.write.format("delta").mode("overwrite").saveAsTable("demo_env.delta_smoke_test")
print("Wrote demo_env.delta_smoke_test")
read_back = spark.table("demo_env.delta_smoke_test").collect()
print("Read back rows:", len(read_back))

In [ ]:
# Simple MLflow run to ensure tracking and model logging works
import mlflow, mlflow.sklearn
from sklearn.linear_model import LogisticRegression
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
import pandas as pd

X, y = make_classification(n_samples=200, n_features=5, random_state=42)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

with mlflow.start_run() as run:
    mlflow.log_param("smoke_test", True)
    model = LogisticRegression(max_iter=200)
    model.fit(X_train, y_train)
    preds = model.predict(X_test)
    acc = accuracy_score(y_test, preds)
    mlflow.log_metric("accuracy", float(acc))
    mlflow.sklearn.log_model(model, "smoke_model")
    run_id = run.info.run_id

print("MLflow run_id:", run_id, "accuracy:", acc)

In [ ]:
# Check that the expected secret scope and keys exist without printing secrets.
scope = "lab-secrets"
keys_to_check = ["DATABRICKS_HOST", "DATABRICKS_TOKEN"]

missing = []
try:
    for k in keys_to_check:
        try:
            _ = dbutils.secrets.get(scope, k)
        except Exception:
            missing.append(k)
except NameError:
    print("dbutils not available in this environment. Secrets check skipped.")
    missing = ["dbutils_unavailable"]

if missing:
    print("Secrets missing or inaccessible:", missing)
else:
    print("All expected secrets present (values not displayed).")

## Troubleshooting

- **Permission errors writing Delta**: ensure your cluster has correct IAM/instance profile or workspace permissions.
- **MLflow URI unexpected**: check `mlflow.get_tracking_uri()`; for Databricks it should point to the workspace tracking server.
- **Secrets missing**: create a secret scope (`lab-secrets`) and add `DATABRICKS_HOST` and `DATABRICKS_TOKEN`.
- **Package install failures**: re-run the install cell or attach a cluster with the recommended Databricks ML runtime.

## Next steps after environment verification

1. Open `01-Feature-Engineering-and-Point-in-Time.ipynb` and run the top cell to install libraries.
2. Run Lab 1 in `labs/lab1_point_in_time.md` following the notebook.
3. If any checks failed, fix them now (secrets, cluster runtime, or package installs) before proceeding.